# 📘 02_metadata_tables.ipynb

## 1. Introduction
This notebook sets up the **metadata tables** needed for Lakehouse Monitoring.  
In **v0.1**, we will only configure the `monitors_control` table to register datasets for monitoring.  
👉 No custom metrics or templates will be included yet.

In [0]:
dbutils.widgets.text("catalog", "dbdemos_steventan", "Catalog")
dbutils.widgets.text("admin_schema", "monitoring_admin", "Admin Schema")
dbutils.widgets.text("out_schema", "lakehouse_monitoring_demo_results", "Output Schema")
dbutils.widgets.text("assets_dir_base", "/Workspace/Users/steven.tan@databricks.com/", "Assets Dir Base")
dbutils.widgets.text("data_schema",    "lakehouse_monitoring",              "Data Schema")

catalog = dbutils.widgets.get("catalog")
admin_schema = dbutils.widgets.get("admin_schema")
out_schema = dbutils.widgets.get("out_schema")
assets_dir_base = dbutils.widgets.get("assets_dir_base")
data_schema = dbutils.widgets.get("data_schema")

## 2. Metadata Tables in Lakehouse Monitoring
Lakehouse Monitoring uses three metadata tables under the **`monitoring_admin` schema**:
1. **`monitors_control`** → Defines which base tables are monitored and where results are written.  
2. **`metric_bindings`** → (Not used in v0.1) Maps monitoring metrics to templates.  
3. **`metric_templates`** → (Not used in v0.1) Stores reusable metric definitions and thresholds.  

For this first version, we will **only work with `monitors_control`**.

## 3) `monitors_control` — Full Schema (v0.1)

This table registers each dataset to be monitored and controls *how/when/where* the monitor runs.

### A. Identity (what to monitor)
| Column | Type | Required | Default | Notes |
|---|---|---:|---|---|
| `table_catalog` | STRING | ✅ | `${catalog}` | Catalog of the source table. |
| `table_schema`  | STRING | ✅ | — | Schema of the source table. |
| `table_name`    | STRING | ✅ | — | Table/view name to monitor. |

### B. Profiling configuration (time series)
| Column | Type | Required | Default | Notes |
|---|---|---:|---|---|
| `profile_type`  | STRING | ✅ | `TimeSeries` | Use time-series profiling. |
| `timestamp_col` | STRING |  | `created_at` | Event time column in the source table. |
| `granularities` | ARRAY\<STRING> |  | `['1 day']` | Windows to aggregate (e.g., `['1 hour','1 day']`). |
| `enable_cdf`    | BOOLEAN |  | `true` | Enable Delta Change Data Feed if present. |

### C. Optional analysis context
| Column | Type | Required | Default | Notes |
|---|---|---:|---|---|
| `baseline_table` | STRING |  | — | Optional FQN for baseline comparison. |
| `slicing_exprs`  | ARRAY\<STRING> |  | — | Optional slice expressions (e.g., `"region"`, `"status='ACTIVE'"`). |

### D. (Reserved) ML/Prediction quality (not used in v0.1)
| Column | Type | Required | Default | Notes |
|---|---|---:|---|---|
| `problem_type`  | STRING |  | — | `classification` / `regression` etc. |
| `prediction_col`| STRING |  | — | Prediction column name. |
| `label_col`     | STRING |  | — | Ground-truth label column. |
| `model_id_col`  | STRING |  | — | Optional model identifier column. |

### E. Output & assets
| Column | Type | Required | Default | Notes |
|---|---|---:|---|---|
| `output_schema_name` | STRING | ✅ | `${catalog}.${out_schema}` | Where monitor writes profile tables. |
| `assets_dir`         | STRING |  | `${assets_dir_base}` | DBFS/Workspace folder for monitor assets. |

### F. Orchestration & notifications
| Column | Type | Required | Default | Notes |
|---|---|---:|---|---|
| `schedule_cron` | STRING |  | `0 0 * * * ?` | Quartz cron (default: **hourly**). |
| `schedule_tz`   | STRING |  | `Asia/Singapore` | Timezone for the schedule. |
| `notifications_on_failure` | ARRAY\<STRING> |  | — | Email list to notify on failure. |
| `enabled`       | BOOLEAN | ✅ | `true` | Flip to pause/resume monitoring. |

> **v0.1 scope**: We use sections **A, B, E, F**.  
> Sections **C** and **D** are optional/reserved for future parts of the series.

In [0]:
def esc(s: str) -> str:
    return s.replace("'", "''")

# Ensure schemas exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{admin_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{out_schema}")

# Use a CONSTANT default for assets_dir = widget base (no functions!)
ddl = f"""
CREATE OR REPLACE TABLE {esc(catalog)}.{esc(admin_schema)}.monitors_control (
  table_catalog      STRING NOT NULL DEFAULT '{esc(catalog)}',
  table_schema       STRING NOT NULL,
  table_name         STRING NOT NULL,

  profile_type       STRING NOT NULL DEFAULT 'TimeSeries',
  timestamp_col      STRING DEFAULT 'created_at',
  granularities      ARRAY<STRING> DEFAULT array('1 day'),

  baseline_table     STRING,
  slicing_exprs      ARRAY<STRING>,

  problem_type       STRING,
  prediction_col     STRING,
  label_col          STRING,
  model_id_col       STRING,

  output_schema_name STRING NOT NULL DEFAULT '{esc(catalog)}.{esc(out_schema)}',
  assets_dir         STRING DEFAULT '{esc(assets_dir_base)}',  -- <<< from widget

  schedule_cron      STRING DEFAULT '0 0 * * * ?',
  schedule_tz        STRING DEFAULT 'Asia/Singapore',
  notifications_on_failure ARRAY<STRING>,
  enable_cdf         BOOLEAN DEFAULT true,
  enabled            BOOLEAN NOT NULL DEFAULT true
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'enabled')
"""

spark.sql(ddl)

print(f"✅ monitors_control created under {catalog}.{admin_schema}")


## 4. Populating `monitors_control`
In this notebook, we insert rows for:
- **policies**  
- **claims**  
- **premium_billing**  

These point to the synthetic data generated in **01_generate_sample_data.ipynb**.

In [0]:
def assets_dir_for(schema: str, table: str) -> str:
    return f"{assets_dir_base}/{schema}/{table}"

# Build the SQL INSERT string
sql = f"""
INSERT INTO {catalog}.{admin_schema}.monitors_control
(table_catalog,
 table_schema,
 table_name,
 profile_type,
 timestamp_col,
 granularities,
 baseline_table,
 slicing_exprs,
 problem_type,
 prediction_col,
 label_col,
 model_id_col,
 output_schema_name,
 assets_dir,
 schedule_cron,
 schedule_tz,
 notifications_on_failure,
 enable_cdf,
 enabled
)
VALUES
('{catalog}', '{data_schema}', 'policies',
 'TimeSeries', 'created_at', array('1 day'),
 NULL, NULL,
 NULL, NULL, NULL, NULL,
 '{catalog}.{out_schema}', '{assets_dir_for("lakehouse_monitoring","policies")}',
 '0 0 * * * ?', 'Asia/Singapore', array(), true, true),

('{catalog}', '{data_schema}', 'claims',
 'TimeSeries', 'reported_at', array('1 day'),
 NULL, NULL,
 NULL, NULL, NULL, NULL,
 '{catalog}.{out_schema}', '{assets_dir_for("lakehouse_monitoring","claims")}',
 '0 0 * * * ?', 'Asia/Singapore', array(), true, true),

('{catalog}', '{data_schema}', 'premium_billing',
 'TimeSeries', 'generated_at', array('1 day'),
 NULL, NULL,
 NULL, NULL, NULL, NULL,
 '{catalog}.{out_schema}', '{assets_dir_for("lakehouse_monitoring","premium_billing")}',
 '0 0 * * * ?', 'Asia/Singapore', array(), true, true)
;
"""

spark.sql(sql)

## 5. What’s Next
At this point:
- Data is available in the Lakehouse (`policies`, `claims`, `premium_billing`).  
- Metadata (`monitors_control`) is ready.  

👉 In the next notebook, we will show how to **run Lakehouse Monitoring** against these tables and explore the results.